# PnLCalib on SoccerNet GSR clips (Kaggle)
Same as `pnlcalib_clip04.ipynb`, but for every clip in the attached dataset (SNGS-028, SNGS-043).

Before running: **Settings → Accelerator → GPU T4 x2**, **Settings → Internet → on**,
and `soccernet_frames.zip` attached as a dataset (**Add Input**). Inside it: `<clip>/img1/000001.jpg …`.

Output per clip: `/kaggle/working/pnlcalib_raw_<clip>.json` + 4 overlay images. ~0.65 s per frame,
so ~8 min per 750-frame clip. Each clip's JSON is saved as soon as that clip is done.
**Download the outputs before stopping the session** (they vanish with a draft session).
The cameras are in **SoccerNet's frame** (origin = centre spot, 105×68 m pitch, z points down),
the same frame as the GSR labels.

In [ ]:
# 1. Get PnLCalib + the single-view weights (main broadcast camera). Kept in /kaggle/temp so
#    the large weight files don't end up in the notebook's output.
!git clone -q https://github.com/mguti97/PnLCalib.git /kaggle/temp/PnLCalib
!pip install -q lsq-ellipse shapely
!wget -q -P /kaggle/temp/PnLCalib/weights https://github.com/mguti97/PnLCalib/releases/download/v1.0.0/SV_kp
!wget -q -P /kaggle/temp/PnLCalib/weights https://github.com/mguti97/PnLCalib/releases/download/v1.0.0/SV_lines
!ls -lh /kaggle/temp/PnLCalib/weights
!nvidia-smi -L

In [ ]:
# 2. Load the two networks (keypoints + line ends).
import glob, json, os, sys, time
os.chdir('/kaggle/temp/PnLCalib'); sys.path.insert(0, '.')

import cv2, numpy as np, torch, yaml
import torchvision.transforms as T
import matplotlib.pyplot as plt

import inference as pnl  # the repo's inference.py; its command-line part doesn't run on import
from model.cls_hrnet import get_cls_net
from model.cls_hrnet_l import get_cls_net as get_cls_net_l
from utils.utils_calib import FramebyFrameCalib

device = 'cuda:0'
pnl.device = device                     # pnl.inference() reads these two as globals
pnl.transform2 = T.Resize((540, 960))   # frames are shrunk to 960x540 before the networks

def load(get_net, cfg, weights):
    net = get_net(yaml.safe_load(open(cfg)))
    net.load_state_dict(torch.load(weights, map_location=device))
    return net.to(device).eval()

model = load(get_cls_net, 'config/hrnetv2_w48.yaml', 'weights/SV_kp')
model_l = load(get_cls_net_l, 'config/hrnetv2_w48_l.yaml', 'weights/SV_lines')
print('models loaded on', device)

In [ ]:
# 3. Calibrate every frame of every clip. Same thresholds and PnL refinement as the repo's README command.
from collections import defaultdict
paths = sorted(glob.glob('/kaggle/input/**/img1/*.jpg', recursive=True))
assert paths, 'no <clip>/img1/*.jpg under /kaggle/input: attach the frames dataset (Add Input)'
clips = defaultdict(list)
for p in paths:
    clips[p.split('/')[-3]].append(p)  # .../SNGS-028/img1/000001.jpg -> SNGS-028
print({c: len(f) for c, f in clips.items()})

all_results = {}
for clip, frames in clips.items():
    h, w = cv2.imread(frames[0]).shape[:2]
    cam = FramebyFrameCalib(iwidth=w, iheight=h, denormalize=True)  # params come out in full-size pixels
    results, t0 = [], time.time()
    for i, path in enumerate(frames):
        res = pnl.inference(cam, cv2.imread(path), model, model_l, 0.3434, 0.7867, True)
        results.append({'frame': int(os.path.basename(path).split('.')[0]),  # SoccerNet frames start at 1
                        'ok': res is not None,
                        'rep_err_px': None if res is None else float(res['rep_err']),
                        'cam_params': None if res is None else res['cam_params']})
        if i % 50 == 0:
            print(f'{clip} {i:4d}/{len(frames)}  frame {results[-1]["frame"]:4d}  ok={results[-1]["ok"]}  {time.time() - t0:.0f} s')
    out = {'clip': clip, 'image_size': [w, h], 'method': 'PnLCalib SV_kp + SV_lines, pnl_refine',
           'world_frame': 'SoccerNet: origin centre spot, 105x68 m, z down', 'frames': results}
    json.dump(out, open(f'/kaggle/working/pnlcalib_raw_{clip}.json', 'w'), indent=1, default=float)
    errs = [r['rep_err_px'] for r in results if r['ok']]
    print(f'{clip} done in {time.time() - t0:.0f} s: {len(errs)}/{len(results)} frames calibrated, '
          f'self-reported error median {np.median(errs):.1f} px, max {np.max(errs):.1f} px')
    all_results[clip] = (frames, results)

In [ ]:
# 4. Look before trusting: draw PnLCalib's pitch (blue) on 4 frames per clip.
#    Reads the saved JSONs, so it also covers clips calibrated in an earlier run of cell 3.
for out_file in sorted(glob.glob('/kaggle/working/pnlcalib_raw_*.json')):
    saved = json.load(open(out_file))
    clip = saved['clip']
    frames = sorted(glob.glob(f'/kaggle/input/**/{clip}/img1/*.jpg', recursive=True))
    by_frame = {r['frame']: r for r in saved['frames']}
    fig, axes = plt.subplots(2, 2, figsize=(20, 11.5))
    for ax, n in zip(axes.flat, [1, 250, 500, 750]):
        img = cv2.imread(next(p for p in frames if os.path.basename(p) == f'{n:06d}.jpg'))
        r = by_frame[n]
        if r['ok']:
            img = pnl.project(img, pnl.projection_from_cam_params(r))
        cv2.imwrite(f'/kaggle/working/overlay_{clip}_{n:06d}.jpg', img)
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); ax.set_axis_off()
        ax.set_title(f'{clip} frame {n}: ' + (f"self-reported {r['rep_err_px']:.1f} px" if r['ok'] else 'FAILED'))
    plt.tight_layout(); plt.show()